<a href="https://colab.research.google.com/github/Eylz-Liu/GSE-M-thode-TAUX/blob/main/Etude_GSE_PRS_Juin_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GSE PRS Proba historique


Section 1 - PerfAction

In [ ]:
!pip install yfinance pandas matplotlib seaborn --quiet

In [1]:
# All package
import yfinance as yf
import pandas as pd
import numpy as np

In [18]:
# Télécharger l'historique depuis 1999
# CAC40
cac40 = yf.download(
    "PX1GR.PA",          #^FCHI
    start="1999-01-01",
    auto_adjust=True,
    progress=False
)

# EURO STOXX 50
eurostoxx50 = yf.download(
    "^STOXX50E",
    start="1999-01-01",
    auto_adjust=True,
    progress=False
)

# Dernier cours de chaque mois
# cac40_monthly = cac40['Close'].resample('ME').last()
# eurostoxx50_monthly = eurostoxx50['Close'].resample('ME').last()


cac40_monthly = (
    cac40[['Close']]
    .groupby(pd.Grouper(freq='ME'))
    .last()
)

eurostoxx50_monthly = (
    eurostoxx50[['Close']]
    .groupby(pd.Grouper(freq='ME'))
    .last()
)



# Fusion
df = pd.concat( [cac40_monthly, eurostoxx50_monthly], axis=1 )

df.columns = ['CAC40', 'EUROSTOXX50']

# Rendement mensuel
# df['CAC40_Return_%'] = df['CAC40'].pct_change() * 100
# df['EUROSTOXX50_Return_%'] = df['EUROSTOXX50'].pct_change() * 100

# Rendements logarithmiques
df['CAC40_Log_Return'] = np.log( df['CAC40'] / df['CAC40'].shift(1) )
df['EUROSTOXX50_Log_Return'] = np.log( df['EUROSTOXX50'] / df['EUROSTOXX50'].shift(1) )

# En %
df['CAC40_Log_Return_%'] = df['CAC40_Log_Return'] * 100
df['EUROSTOXX50_Log_Return_%'] = df['EUROSTOXX50_Log_Return'] * 100

# Rendement logarithmique sur 12 mois
import numpy as np

df['CAC40_Log_Return_12M'] = np.log(
    df['CAC40'] / df['CAC40'].shift(12)
)

df['EUROSTOXX50_Log_Return_12M'] = np.log(
    df['EUROSTOXX50'] / df['EUROSTOXX50'].shift(12)
)

# en pourcentage
df['CAC40_Log_Return_12M_%'] = (
    df['CAC40_Log_Return_12M'] * 100
)

df['EUROSTOXX50_Log_Return_12M_%'] = (
    df['EUROSTOXX50_Log_Return_12M'] * 100
)


print(df.head())
print(df.tail())

                  CAC40  EUROSTOXX50  CAC40_Log_Return  \
Date                                                     
1999-01-31  5903.120117          NaN               NaN   
1999-02-28  5682.729980          NaN         -0.038049   
1999-03-31  5829.040039          NaN          0.025421   
1999-04-30  6122.060059          NaN          0.049046   
1999-05-31  6078.589844          NaN         -0.007126   

            EUROSTOXX50_Log_Return  CAC40_Log_Return_%  \
Date                                                     
1999-01-31                     NaN                 NaN   
1999-02-28                     NaN           -3.804930   
1999-03-31                     NaN            2.542058   
1999-04-30                     NaN            4.904632   
1999-05-31                     NaN           -0.712592   

            EUROSTOXX50_Log_Return_%  CAC40_Log_Return_12M  \
Date                                                         
1999-01-31                       NaN                   NaN   


In [5]:
# Information of DF shape
print("Dimensions du DataFrame :", df.shape)
print("Colonnes :")
for col in df.columns:
    print("-", col)

Dimensions du DataFrame : (331, 10)
Colonnes :
- CAC40
- EUROSTOXX50
- CAC40_Log_Return
- EUROSTOXX50_Log_Return
- CAC40_Log_Return_%
- EUROSTOXX50_Log_Return_%
- CAC40_Log_Return_12M
- EUROSTOXX50_Log_Return_12M
- CAC40_Log_Return_12M_%
- EUROSTOXX50_Log_Return_12M_%


In [6]:

# Moyenne mensuelle (%)
cac_mean_monthly = df['CAC40_Log_Return_%'].mean()
euro_mean_monthly = df['EUROSTOXX50_Log_Return_%'].mean()

# Ecart-type mensuel
cac_std_monthly = df['CAC40_Log_Return_%'].std()
euro_std_monthly = df['EUROSTOXX50_Log_Return_%'].std()

# Annualisation
cac_mean_annual = cac_mean_monthly * 12
euro_mean_annual = euro_mean_monthly * 12

cac_vol_annual = cac_std_monthly * np.sqrt(12)
euro_vol_annual = euro_std_monthly * np.sqrt(12)

print("CAC40")
print(f"  Ecart-type mensuel      : {cac_std_monthly:.4f}")
print(f"  Volatilité annualisée   : {cac_vol_annual:.4f}")
print(f"  Volatilité annualisée % : {cac_vol_annual*100:.2f}%")

print("\nEURO STOXX 50")
print(f"  Ecart-type mensuel      : {euro_std_monthly:.4f}")
print(f"  Volatilité annualisée   : {euro_vol_annual:.4f}")
print(f"  Volatilité annualisée % : {euro_vol_annual*100:.2f}%")

# Tableau récapitulatif
stats = pd.DataFrame({
    'Indice': ['CAC40', 'EUROSTOXX50'],
    'Moyenne_Mensuelle_%': [cac_mean_monthly, euro_mean_monthly],
    'Moyenne_Annualisee_%': [cac_mean_annual, euro_mean_annual],
    'Volatilite_Mensuelle_%': [cac_std_monthly, euro_std_monthly],
    'Volatilite_Annualisee_%': [cac_vol_annual, euro_vol_annual]
})

print(stats.round(3))


CAC40
  Ecart-type mensuel      : 5.3220
  Volatilité annualisée   : 18.4359
  Volatilité annualisée % : 1843.59%

EURO STOXX 50
  Ecart-type mensuel      : 5.0943
  Volatilité annualisée   : 17.6471
  Volatilité annualisée % : 1764.71%
        Indice  Moyenne_Mensuelle_%  Moyenne_Annualisee_%  \
0        CAC40                0.306                 3.669   
1  EUROSTOXX50                0.184                 2.210   

   Volatilite_Mensuelle_%  Volatilite_Annualisee_%  
0                   5.322                   18.436  
1                   5.094                   17.647  


# Moyen log rendement sur 12Mois

In [7]:

# Moyenne mensuelle (%)
cac_mean_monthly = df['CAC40_Log_Return_12M_%'].mean()
euro_mean_monthly = df['EUROSTOXX50_Log_Return_12M_%'].mean()

# Ecart-type mensuel
cac_std_monthly_ = df['CAC40_Log_Return_12M_%'].std()
euro_std_monthly = df['EUROSTOXX50_Log_Return_12M_%'].std()

# Annualisation
cac_mean_annual = cac_mean_monthly * 12
euro_mean_annual = euro_mean_monthly * 12

cac_vol_annual = cac_std_monthly * np.sqrt(12)
euro_vol_annual = euro_std_monthly * np.sqrt(12)

print("CAC40")
print(f"  Ecart-type mensuel      : {cac_std_monthly:.4f}")
print(f"  Volatilité annualisée   : {cac_vol_annual:.4f}")
print(f"  Volatilité annualisée % : {cac_vol_annual*100:.2f}%")

print("\nEURO STOXX 50")
print(f"  Ecart-type mensuel      : {euro_std_monthly:.4f}")
print(f"  Volatilité annualisée   : {euro_vol_annual:.4f}")
print(f"  Volatilité annualisée % : {euro_vol_annual*100:.2f}%")

# Tableau récapitulatif
stats = pd.DataFrame({
    'Indice': ['CAC40', 'EUROSTOXX50'],
    'Moyenne_Mensuelle_%': [cac_mean_monthly, euro_mean_monthly],
    'Moyenne_Annualisee_%': [cac_mean_annual, euro_mean_annual],
    'Volatilite_Mensuelle_%': [cac_std_monthly, euro_std_monthly],
    'Volatilite_Annualisee_%': [cac_vol_annual, euro_vol_annual]
})

print(stats.round(3))


CAC40
  Ecart-type mensuel      : 5.3220
  Volatilité annualisée   : 18.4359
  Volatilité annualisée % : 1843.59%

EURO STOXX 50
  Ecart-type mensuel      : 17.9978
  Volatilité annualisée   : 62.3463
  Volatilité annualisée % : 6234.63%
        Indice  Moyenne_Mensuelle_%  Moyenne_Annualisee_%  \
0        CAC40                3.516                42.187   
1  EUROSTOXX50                1.708                20.499   

   Volatilite_Mensuelle_%  Volatilite_Annualisee_%  
0                   5.322                   18.436  
1                  17.998                   62.346  


# Kurtosis & Asymétrie

In [8]:
# Asymétrie (Excel COEFFICIENT.ASYMETRIE)
cac_skew = df['CAC40_Log_Return_12M_%'].skew()
euro_skew = df['EUROSTOXX50_Log_Return_12M_%'].skew()

# Kurtosis (Excel KURTOSIS)
cac_kurt = df['CAC40_Log_Return_12M_%'].kurt()
euro_kurt = df['EUROSTOXX50_Log_Return_12M_%'].kurt()

print("CAC40")
print(f"Asymétrie (Skewness) : {cac_skew:.4f}")
print(f"Kurtosis            : {cac_kurt:.4f}")

print("\nEURO STOXX 50")
print(f"Asymétrie (Skewness) : {euro_skew:.4f}")
print(f"Kurtosis            : {euro_kurt:.4f}")

CAC40
Asymétrie (Skewness) : -0.7165
Kurtosis            : -0.3381

EURO STOXX 50
Asymétrie (Skewness) : -1.1566
Kurtosis            : 1.7821


In [ ]:
stats = pd.DataFrame({
    'Indice': ['CAC40', 'EUROSTOXX50'],
    'Moyenne_Annualisee_%': [
        cac_mean_annual,
        euro_mean_annual
    ],
    'Volatilite_Annualisee_%': [
        cac_vol_annual,
        euro_vol_annual
    ],
    'Skewness': [
        df['CAC40_Log_Return_12M_%'].skew(),
        df['EUROSTOXX50_Log_Return_12M_%'].skew()
    ],
    'Kurtosis': [
        df['CAC40_Log_Return_12M_%'].kurt(),
        df['EUROSTOXX50_Log_Return_12M_%'].kurt()
    ]
})

print(stats.round(4))

        Indice  Moyenne_Annualisee_%  Volatilite_Annualisee_%  Skewness  \
0        CAC40               25.6938                  17.4249   -0.8040   
1  EUROSTOXX50               20.4780                  62.3407   -1.1569   

   Kurtosis  
0    0.5715  
1    1.7831  


# Exporter

In [17]:
from google.colab import files

# Nom du fichier à télécharger
file_name = "CAC40_EuroStoxx50_Historique.xlsx"

# Création du fichier Excel
df.to_excel(
    file_name,
    sheet_name="Historique",
    index=True,
    engine="openpyxl"
)

print(f"Fichier créé : {file_name}")

# Téléchargement automatique
files.download(file_name)

Fichier créé : CAC40_EuroStoxx50_Historique.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>